In [1]:
import os
import pandas as pd
from lxml import etree

In [2]:
year= 2025
empresa="Indux"
sentido="Emitidos"
def list_files_in_directory(directory):
    # Lista para almacenar detalles de los archivos
    file_list = []
    # Recorrer el directorio y subdirectorios
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.xml'):
                # Ruta completa del archivo
                file_path = os.path.join(root, file)

                try:
                    # Cargar el archivo XML
                    tree = etree.parse(file_path)
                    root_element = tree.getroot()
                
                    version = root_element.get("Version", root_element.get("version"))
                    
                    if version == "3.3":
                        namespaces = {
                            'cfdi': 'http://www.sat.gob.mx/cfd/3',
                            'nomina12' : 'http://www.sat.gob.mx/nomina12',
                            'tfd': 'http://www.sat.gob.mx/TimbreFiscalDigital',
                        }
                    elif version == "4.0":
                        namespaces = {
                            'cfdi': 'http://www.sat.gob.mx/cfd/4',
                            'nomina12' : 'http://www.sat.gob.mx/nomina12',
                            'tfd': 'http://www.sat.gob.mx/TimbreFiscalDigital',
                        }
                    else:
                        print(f"Versión de CFDI desconocida en archivo: {file_path}")
                        continue

                    # Extraer la información principal del CFDI (root)
                    metodo_pago = root_element.get('MetodoPago', 'N/A')
                    forma_pago = root_element.get('FormaPago', 'N/A')
                    moneda = root_element.get('Moneda', 'N/A')
                    lugarex = root_element.get('LugarExpedicion', 'N/A')
                    ffecha = root_element.get('Fecha', 'N/A')
                    fcondiciones = root_element.get('CondicionesDePago', 'N/A')

                    # Extraer los detalles del receptor y emisor
                    receptord = root_element.find('.//cfdi:Receptor', namespaces)
                    emisord = root_element.find('.//cfdi:Emisor', namespaces)

                    rfce = emisord.get('Rfc', 'N/A') if emisord is not None else 'N/A'
                    nombree = emisord.get('Nombre', 'N/A') if emisord is not None else 'N/A'
                    rege = emisord.get('RegimenFiscalReceptor', 'N/A') if emisord is not None else 'N/A'

                    rfcr = receptord.get('Rfc', 'N/A') if receptord is not None else 'N/A'
                    nombrer = receptord.get('Nombre', 'N/A') if receptord is not None else 'N/A'
                    regr = receptord.get('RegimenFiscalReceptor', 'N/A') if receptord is not None else 'N/A'


                    # Buscar el nodo Receptor dentro del complemento de nómina
                    nomina_receptor = root_element.find('.//nomina12:Receptor', namespaces)
                    num_empleado = nomina_receptor.get('NumEmpleado', 'N/A') if nomina_receptor is not None else 'N/A'
                    
                    
                    # Percepciones, Deducciones y OtrosPagos
                    nomina = root_element.find('.//nomina12:Nomina', namespaces)
                    ffechap = nomina.get('FechaPago', 'N/A')
                    ffechapi = nomina.get('FechaInicialPago', 'N/A')
                    ffechapf = nomina.get('FechaFinalPago', 'N/A')
                    # Extraer el UUID del complemento de timbre fiscal digital (TimbreFiscalDigital)
                    timbre_fiscal = root_element.find('.//tfd:TimbreFiscalDigital', namespaces)
                    uuid = timbre_fiscal.get('UUID', 'N/A') if timbre_fiscal is not None else 'N/A'

                    # Extraer la fecha de timbrado
                    fecha_timbrado = timbre_fiscal.get('FechaTimbrado', 'N/A') if timbre_fiscal is not None else 'N/A'

                    
                    if nomina is not None:
                        # Iterar sobre cada nodo <nomina12:Nomina>
                        nomina_nodos = root_element.findall('.//nomina12:Nomina', namespaces)
                        for nodo_nomina in nomina_nodos:
                            ffechap = nodo_nomina.get('FechaPago', 'N/A')
                            ffechapi = nodo_nomina.get('FechaInicialPago', 'N/A')
                            ffechapf = nodo_nomina.get('FechaFinalPago', 'N/A')
                            
                            # Extraer percepciones
                            percepciones_nodos = nodo_nomina.findall('.//nomina12:Percepciones', namespaces)
                            for nodo_percepciones in percepciones_nodos:
                                percepciones = nodo_percepciones.findall('.//nomina12:Percepcion', namespaces)
                                for percepcion in percepciones:
                                    Clave = percepcion.get('Clave', 'N/A')
                                    Tipo_p = percepcion.get('TipoPercepcion', 'N/A')
                                    Concepto = percepcion.get('Concepto', 'N/A')
                                    ImporteGravado = percepcion.get('ImporteGravado', 0)
                                    ImporteExento = percepcion.get('ImporteExento', 0)
                                    
                                    # Agregar los detalles del archivo con percepciones
                                    file_list.append({
                                        "Archivo": file,
                                        "UUID": uuid,
                                        "MetodoPago": metodo_pago,
                                        "FormaPago": forma_pago,
                                        "Fecha Pago": ffechap,
                                        "Fecha Pago I": ffechapi,
                                        "Fecha Pago F": ffechapf,
                                        "ExpedicionL": lugarex,
                                        "Moneda": moneda,
                                        "Fecha": ffecha,
                                        "CondicionesDePago": fcondiciones,
                                        "RFC Emisor": rfce,
                                        "Nombre Emisor": nombree,
                                        "Régimen Emisor": rege,
                                        "RFC Receptor": rfcr,
                                        "Nombre Receptor": nombrer,
                                        "Régimen Receptor": regr,
                                        "Empleado": num_empleado,
                                        "Tipo Concepto": "Percepcion",
                                        "Clave": Clave,
                                        "Tipo_": Tipo_p,
                                        "Concepto": Concepto,
                                        "Importe": 0,
                                        "Importe Gravado": ImporteGravado,
                                        "Importe Exento": ImporteExento
                                    })
                    
                            # Extraer deducciones
                            deducciones = nodo_nomina.findall('.//nomina12:Deducciones/nomina12:Deduccion', namespaces)
                            for deduccion in deducciones:
                                Clave = deduccion.get('Clave', 'N/A')
                                Tipo_d = deduccion.get('TipoDeduccion', 'N/A')
                                Concepto = deduccion.get('Concepto', 'N/A')
                                Importe = deduccion.get('Importe', 0)
                    
                                # Agregar los detalles del archivo con deducciones
                                file_list.append({
                                    "Archivo": file,
                                    "UUID": uuid,
                                    "MetodoPago": metodo_pago,
                                    "Fecha Pago": ffechap,
                                    "Fecha Pago I": ffechapi,
                                    "Fecha Pago F": ffechapf,
                                    "FormaPago": forma_pago,
                                    "ExpedicionL": lugarex,
                                    "Moneda": moneda,
                                    "Fecha": ffecha,
                                    "CondicionesDePago": fcondiciones,
                                    "RFC Emisor": rfce,
                                    "Nombre Emisor": nombree,
                                    "Régimen Emisor": rege,
                                    "RFC Receptor": rfcr,
                                    "Nombre Receptor": nombrer,
                                    "Régimen Receptor": regr,
                                    "Empleado": num_empleado,
                                    "Tipo Concepto": "Deduccion",
                                    "Clave": Clave,
                                    "Tipo_": Tipo_d,
                                    "Concepto": Concepto,
                                    "Importe": Importe,
                                    "Importe Gravado": 0,
                                    "Importe Exento": 0
                                })
                    
                            # Extraer otros pagos
                            otros_pagos = nodo_nomina.findall('.//nomina12:OtrosPagos/nomina12:OtroPago', namespaces)
                            for otro_pago in otros_pagos:
                                Clave = otro_pago.get('Clave', 'N/A')
                                Tipo_o = otro_pago.get('TipoPercepcion', 'N/A')
                                Concepto = otro_pago.get('Concepto', 'N/A')
                                Importe = otro_pago.get('Importe', 0)
                    
                                # Agregar los detalles del archivo con otros pagos
                                file_list.append({
                                    "Archivo": file,
                                    "UUID": uuid, 
                                    "MetodoPago": metodo_pago,
                                    "Fecha Pago": ffechap,
                                    "Fecha Pago I": ffechapi,
                                    "Fecha Pago F": ffechapf,
                                    "FormaPago": forma_pago,
                                    "ExpedicionL": lugarex,
                                    "Moneda": moneda,
                                    "Fecha": ffecha,
                                    "RFC Emisor": rfce,
                                    "Nombre Emisor": nombree,
                                    "Régimen Emisor": rege,
                                    "RFC Receptor": rfcr,
                                    "Nombre Receptor": nombrer,
                                    "Régimen Receptor": regr,
                                    "Empleado": num_empleado,
                                    "Tipo Concepto": "OtroPago",
                                    "Clave": Clave,
                                    "Concepto": Concepto,
                                    "Importe": Importe,
                                    "Importe Gravado": 0,
                                    "Importe Exento": 0
                                })
                    
                except Exception as e:
                    print(f"Error procesando el archivo {file}: {e}")

    # Retornar la lista de archivos con su detalle
    return file_list

# Especificar el directorio de los archivos XML
directory_path = fr'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2025 XML\Recibidos\Trabajo Nomina'
#directory_path = r'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2024 XML\Descarga 250330\Recibidos\Trabajo Nomina'


In [3]:
d_path = fr'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2025 XML\Recibidos'
#d_path = r'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2024 XML\Descarga 250330\Recibidos'
df_files = list_files_in_directory(directory_path)
df_files_df = pd.DataFrame(df_files)
output_csv_path = os.path.join(d_path, f'D_Nomina {empresa} {year}.csv')
df_files_df.to_csv(output_csv_path ,index = False)
print(f"✅ El archivo se ha guardado en: {output_csv_path}")

✅ El archivo se ha guardado en: C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2025 XML\Recibidos\D_Nomina Indux 2025.csv
